# 01 — Big Data and Spark

**Course:** PySpark from Zero to Production
**Module 1 of 15**

## Learning Objectives
By the end of this notebook you will be able to:
- Explain what "big data" means in practice, not just buzzwords
- Describe the problems that Hadoop MapReduce solved and where it fell short
- Explain what Apache Spark is and why it replaced MapReduce for most workloads
- List the core Spark components (Core, SQL, Streaming, MLlib, GraphX)
- Understand where PySpark fits in the Spark ecosystem

---


## 1.1 What Is "Big Data"?

Big data is usually described with the **3 Vs** (sometimes extended to 5):

| V | Meaning | Example |
|---|---------|---------|
| **Volume** | The sheer amount of data | Terabytes/petabytes of clickstream logs |
| **Velocity** | The speed at which data arrives | Millions of IoT sensor readings per second |
| **Variety** | The different formats/structures | JSON, CSV, images, video, text, Parquet |
| **Veracity** *(extended)* | Data quality and trustworthiness | Noisy sensor data, duplicated records |
| **Value** *(extended)* | The business value locked in the data | Recommendations, fraud detection |

The practical definition that matters for engineers: **big data is data that no longer fits, or no longer processes fast enough, on a single machine.** When a dataset outgrows the RAM and CPU of one computer, you need a *distributed system* — many machines cooperating on the same job.

This is the entire reason distributed computing frameworks like Hadoop and Spark exist.


## 1.2 The Problem With a Single Machine

Imagine you have a 10 TB dataset and a single powerful server with 128 GB RAM.

- You can't load 10 TB into 128 GB of RAM.
- Even reading 10 TB from disk sequentially at ~200 MB/s takes almost 14 hours.
- If the machine crashes mid-job, you start over.

The solution is to **split the data across many machines** and **split the computation across many machines too**, so each machine only processes a small piece, in parallel, and the results are combined. This is called **horizontal scaling** (adding more machines) as opposed to **vertical scaling** (buying a bigger machine).

This is the foundation of every big data framework: **partition the data, distribute the work, combine the results.**


## 1.3 Enter Hadoop and MapReduce

In the early 2000s, Google published papers describing the **Google File System (GFS)** and **MapReduce**. The open-source community built **Apache Hadoop** based on these ideas, consisting of:

- **HDFS (Hadoop Distributed File System)** — stores huge files by splitting them into blocks distributed across many machines.
- **MapReduce** — a programming model with two phases:
  - **Map**: transform each record independently (e.g., extract a word from a line of text).
  - **Reduce**: aggregate the mapped records (e.g., count occurrences of each word).

MapReduce made distributed computation accessible, but it had real pain points:

1. **Disk I/O heavy** — MapReduce writes intermediate results to disk between every Map and Reduce stage. For multi-step pipelines (common in machine learning and iterative algorithms), this is extremely slow.
2. **Verbose programming model** — even simple operations like a `filter` + `groupBy` required writing significant boilerplate Java code.
3. **No good support for interactive queries or iterative algorithms** — every job re-reads data from disk, so trying 10 variations of an experiment means reading the dataset 10 times from disk.


## 1.4 Apache Spark: The Next Generation

Apache Spark was created at UC Berkeley's AMPLab (2009) specifically to fix MapReduce's weaknesses. It became an Apache project in 2013 and is now the de-facto standard engine for large-scale data processing.

**Key ideas that made Spark faster and easier to use:**

1. **In-memory computation** — Spark keeps intermediate data in RAM across stages whenever possible, avoiding repeated disk writes. This alone can make iterative jobs 10-100x faster than equivalent MapReduce jobs.
2. **Lazy evaluation with DAGs** — Spark builds a *Directed Acyclic Graph* of the operations you request, then optimizes the whole plan before running anything. This is covered fully in Notebook 02.
3. **Rich, high-level APIs** — instead of writing raw Map/Reduce boilerplate, you write expressive code:
   ```python
   df.filter(df.age > 30).groupBy("department").count()
   ```
4. **Unified engine** — one engine, many workloads: batch processing, SQL queries, streaming, machine learning, and graph processing.
5. **Multi-language support** — Scala (native), Java, **Python (PySpark)**, R, and SQL.


## 1.5 Spark's Component Stack

Apache Spark is not just one thing — it's a unified stack of libraries sitting on top of a common engine (**Spark Core**):

```
┌─────────────────────────────────────────────────────────────┐
│                     Spark SQL / DataFrames                    │  Structured data, SQL queries
├───────────────┬───────────────┬───────────────┬───────────────┤
│  Spark        │   MLlib       │   GraphX      │  Structured    │
│  Streaming    │  (Machine     │  (Graph       │  Streaming     │
│  (legacy)     │  Learning)    │  Processing)  │                │
├───────────────┴───────────────┴───────────────┴───────────────┤
│                          Spark Core                            │
│         (RDDs, task scheduling, memory management)             │
└─────────────────────────────────────────────────────────────┘
```

- **Spark Core** — the foundation: RDDs (Resilient Distributed Datasets), task scheduling, fault tolerance, memory management.
- **Spark SQL / DataFrames** — structured data processing with a SQL-like API. This is what you'll use 90% of the time in modern PySpark work (Notebooks 06-10).
- **Spark MLlib** — scalable machine learning algorithms (Notebook 13).
- **Structured Streaming** — real-time processing of streaming data (mentioned briefly in this course; a large topic on its own).
- **GraphX** — graph-parallel computation (Scala-only, not covered in this course).

### Where does PySpark fit?

**PySpark** is the Python API for Apache Spark. Under the hood, PySpark code is translated into JVM instructions that run on the actual Spark engine (written in Scala, running on the JVM). This means:

- You write familiar, readable Python.
- You get the full performance of the distributed JVM-based Spark engine.
- There is a small serialization overhead when moving data between the Python and JVM processes — one of the reasons DataFrame/SQL APIs (which stay inside the JVM) are preferred over raw RDDs with Python UDFs (Notebook 11 covers this).


## 1.6 When Should You Actually Use Spark?

Spark is powerful, but it's not always the right tool. A quick rule of thumb:

| Situation | Recommended tool |
|---|---|
| Dataset fits comfortably in memory on one machine (< a few GB) | pandas / Polars |
| Dataset is tens of GB to many TB, or growing | **Spark** |
| You need a single SQL query on a data warehouse | Warehouse SQL engine (Snowflake, BigQuery) |
| You need distributed ETL, batch processing at scale | **Spark** |
| You need real-time streaming analytics | Spark Structured Streaming / Flink / Kafka Streams |
| You need distributed machine learning training | Spark MLlib or specialized frameworks (e.g. Horovod) |

A common mistake for beginners is reaching for Spark on datasets that fit in memory — the distributed overhead (task scheduling, serialization, network shuffles) can make Spark **slower** than pandas for small data. Spark shines when the data genuinely doesn't fit on one machine.


In [ ]:
# Quick sanity check: no Spark needed yet, just Python.
# This cell just confirms your notebook environment executes code correctly.
print("Environment check: OK")
print("In the next notebook, we'll open the hood on Spark's architecture.")


## 1.7 Summary

- Big data = data too large or too fast-moving for a single machine to handle efficiently.
- Hadoop MapReduce pioneered distributed processing but suffered from heavy disk I/O and verbose code.
- Spark solves this with in-memory computation, lazy evaluation with optimized execution plans, and high-level, expressive APIs.
- Spark is a unified engine: Spark Core + Spark SQL/DataFrames + MLlib + Streaming + GraphX.
- PySpark is the Python API that lets you drive the Scala/JVM Spark engine with Python code.

## Check Your Understanding
1. What are the 3 core "Vs" of big data?
2. Name two specific weaknesses of Hadoop MapReduce that Spark was designed to fix.
3. True or false: Spark always outperforms pandas, regardless of data size. Explain your answer.
4. Which Spark component would you use to run a SQL query against structured data?

---
**Next:** [02_Spark_Architecture.ipynb](02_Spark_Architecture.ipynb) — Driver, Executors, Cluster Managers, and how a Spark job actually runs.
